In [2]:
"""
kagome_common.py
-----------------
Utilidades compartidas para reproducir los resultados de:

Ahsan, M. "Utility-scale Experimental Quantum Computation with Hardware
Efficient Ansatze and Calibrated Hamiltonian", ACM Trans. Quantum Comput. (2026)

Convención del paper (Sección 3.1, ecuación 1):
    H = J * sum_{(a,b) in bonds} (X_a X_b + Y_a Y_b + Z_a Z_b)

Con esta normalización (factor "4x" mencionado en el paper), la energía de un
singlete en un bond individual es -3 y la de un triplete es +1 (en vez de
-3/4 y +1/4 de la convención estándar S_i . S_j).

Requiere: qiskit >= 2.0, scipy, numpy
    pip install qiskit scipy numpy
"""

import numpy as np
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import RealAmplitudes
from scipy.optimize import minimize


def build_hamiltonian(n_qubits, bonds):
    """
    Construye H = sum_bonds J*(XiXj + YiYj + ZiZj) como SparsePauliOp.

    bonds : lista de tuplas (i, j, J) -- sitios i,j (0-indexed) y su
            acoplamiento J (J=1 para bonds normales, J' para bonds calibrados).
    """
    terms, coeffs = [], []
    for (i, j, J) in bonds:
        for P in ['X', 'Y', 'Z']:
            label = ['I'] * n_qubits
            label[n_qubits - 1 - i] = P   # Qiskit usa orden little-endian
            label[n_qubits - 1 - j] = P
            terms.append(''.join(label))
            coeffs.append(J)
    return SparsePauliOp(terms, coeffs).simplify()


def exact_gs_energy(H, sparse_threshold=12):
    """
    Diagonalización exacta del Hamiltoniano.
    - Para n_qubits pequeño (<=12): diagonalización densa completa (numpy).
    - Para n_qubits mayor: solo el autovalor más bajo vía Lanczos disperso
      (scipy.sparse.linalg.eigsh), para evitar quedarse sin memoria.
    """
    n = H.num_qubits
    if n <= sparse_threshold:
        mat = H.to_matrix()
        evals = np.linalg.eigvalsh(mat)
        return evals[0]
    else:
        from scipy.sparse.linalg import eigsh
        mat = H.to_matrix(sparse=True)
        evals = eigsh(mat, k=1, which='SA', maxiter=5000, tol=1e-10,
                       return_eigenvectors=False)
        return evals[0]


def real_amplitudes_ansatz(n_qubits, reps=1):
    """
    El ansatz hardware-efficient del paper (Fig. 1b, Fig. 9a):
    capa de Ry(theta) -> escalera lineal de CNOT -> capa final de Ry(theta).
    Corresponde exactamente a qiskit.circuit.library.RealAmplitudes(reps=1,
    entanglement='linear').
    """
    return RealAmplitudes(n_qubits, reps=reps, entanglement='linear')


def vqe_energy(H, n_qubits, reps=1, n_restarts=8, seed=0, maxiter=400,
               method='L-BFGS-B'):
    """
    VQE clásico (statevector, sin ruido) sobre el ansatz de arriba.

    method: 'L-BFGS-B' (rápido para sistemas chicos, usa diferencias finitas)
            'COBYLA'   (más barato por iteración; recomendado para >12 qubits)

    Devuelve (mejor_energia, mejores_parametros, ansatz).
    """
    ansatz = real_amplitudes_ansatz(n_qubits, reps=reps)
    nparams = ansatz.num_parameters
    rng = np.random.default_rng(seed)

    def cost(theta):
        bound = ansatz.assign_parameters(theta)
        sv = Statevector.from_instruction(bound)
        return np.real(sv.expectation_value(H))

    best, best_val = None, np.inf
    for _ in range(n_restarts):
        x0 = rng.uniform(0, 2 * np.pi, size=nparams)
        if method == 'COBYLA':
            res = minimize(cost, x0, method='COBYLA',
                            options={'maxiter': maxiter, 'tol': 1e-8})
        else:
            res = minimize(cost, x0, method='L-BFGS-B',
                            options={'maxiter': maxiter, 'ftol': 1e-12,
                                     'gtol': 1e-10})
        if res.fun < best_val:
            best_val, best = res.fun, res.x
    return best_val, best, ansatz


def bond_energy_map(bonds, n_qubits, theta, ansatz):
    """
    Calcula <psi| (XiXj+YiYj+ZiZj) |psi> bond por bond, para el estado
    preparado por `ansatz` con parámetros `theta`.  Reproduce los mapas de
    energía de bond de la Fig. 1(c) del paper.

    bonds: lista de (i, j, J) -- J se usa solo como referencia/etiqueta,
           el bond individual siempre se evalúa con coeficiente 1
           salvo que se pase explícitamente J.
    """
    bound = ansatz.assign_parameters(theta)
    sv = Statevector.from_instruction(bound)
    out = {}
    for (i, j, J) in bonds:
        Hij = build_hamiltonian(n_qubits, [(i, j, J)])
        out[(i, j)] = np.real(sv.expectation_value(Hij))
    return out


In [3]:
"""
reproduce_6site_fig1.py
------------------------
Reproduce la Figura 1 (subregion de 6 sitios) y la fila de 6 sitios de la
Tabla 1 del paper de Ahsan.

Topologia (Fig. 1a): triangulo defecto (1,3,4) + tres sitios colgantes
(pendant) 0,2,5 unidos por un solo bond cada uno:
    bonds = (0,1), (1,3), (1,4), (3,4), (2,3), (4,5)   J=1 en todos

Hamiltoniano calibrado H_pert (Fig. 1a, ecuacion 1): se sube un bond del
triangulo defecto, (3,4), de J=1 a J'=2.

Resultados esperados (paper, Tabla 1 fila "6 sitios" y Fig. 1c):
    Exacto (ED):                         -10.0
    VQE sin calibrar  <phi|H|phi>:        -9.0   (error ~10%)
    VQE calibrado <psi|H_pert|psi>:       -9.98  (error ~0.2%, J'=2)
"""

from kagome_common import (build_hamiltonian, exact_gs_energy, vqe_energy,
                            bond_energy_map)

n = 6

# --- Hamiltoniano original (todos los bonds J=1) ---
bonds_orig = [(0, 1, 1), (1, 3, 1), (1, 4, 1), (3, 4, 1), (2, 3, 1), (4, 5, 1)]

# --- Hamiltoniano calibrado: J'=2 en el bond (3,4) del triangulo defecto ---
bonds_pert = [(0, 1, 1), (1, 3, 1), (1, 4, 1), (3, 4, 2), (2, 3, 1), (4, 5, 1)]

H_orig = build_hamiltonian(n, bonds_orig)
H_pert = build_hamiltonian(n, bonds_pert)

# 1) Diagonalizacion exacta (benchmark de la Tabla 1)
E_exact = exact_gs_energy(H_orig)
print(f"Energia exacta del estado fundamental de H (ED):  {E_exact:.4f}")
print(f"  -> Paper (Tabla 1, 6 sitios): -10.0\n")

# 2) VQE sin calibrar, sobre H original
val_o, theta_o, ansatz = vqe_energy(H_orig, n, reps=1, n_restarts=12, seed=1)
err_o = abs(val_o - E_exact) / abs(E_exact) * 100
print(f"VQE <phi|H|phi>          (sin calibrar): {val_o:.4f}   error={err_o:.2f}%")
print(f"  -> Paper (Fig. 1c): -9.0   error=10%\n")

# 3) VQE calibrado, sobre H_pert (J'=2)
val_p, theta_p, ansatz_p = vqe_energy(H_pert, n, reps=1, n_restarts=12, seed=2)
err_p = abs(val_p - E_exact) / abs(E_exact) * 100
print(f"VQE <psi|H_pert|psi>     (J'=2 calibrado): {val_p:.4f}   error={err_p:.2f}%")
print(f"  -> Paper (Fig. 1c): -9.98   error=0.2%\n")

# 4) Mapas de energia de bond (Fig. 1c)
print("Mapa de energia de bond, estado sin calibrar |phi>:")
for (i, j), v in bond_energy_map(bonds_orig, n, theta_o, ansatz).items():
    print(f"  e({i},{j}) = {v:.2f}")

print("\nMapa de energia de bond, estado calibrado |psi> (J'=2 en (3,4)):")
for (i, j), v in bond_energy_map(bonds_pert, n, theta_p, ansatz_p).items():
    print(f"  e({i},{j}) = {v:.2f}")
print("\n  -> Paper (Fig. 1c): e(0,1)=-3, e(4,5)=-1.47, e(2,3)=-1.32, e(3,4)=-4.19")
print("     (dimer (0,1) intacto; bond (3,4) reforzado; dimers (2,3)/(4,5)")
print("      'derretidos' en superposicion -- firma de estado tipo spin-liquid)")


Energia exacta del estado fundamental de H (ED):  -10.0000
  -> Paper (Tabla 1, 6 sitios): -10.0

VQE <phi|H|phi>          (sin calibrar): -9.1992   error=8.01%
  -> Paper (Fig. 1c): -9.0   error=10%

VQE <psi|H_pert|psi>     (J'=2 calibrado): -10.1160   error=1.16%
  -> Paper (Fig. 1c): -9.98   error=0.2%

Mapa de energia de bond, estado sin calibrar |phi>:
  e(0,1) = -2.96
  e(1,3) = -0.20
  e(1,4) = 0.04
  e(3,4) = -0.20
  e(2,3) = -2.92
  e(4,5) = -2.96

Mapa de energia de bond, estado calibrado |psi> (J'=2 en (3,4)):
  e(0,1) = -3.00
  e(1,3) = -0.02
  e(1,4) = 0.01
  e(3,4) = -4.33
  e(2,3) = -1.39
  e(4,5) = -1.39

  -> Paper (Fig. 1c): e(0,1)=-3, e(4,5)=-1.47, e(2,3)=-1.32, e(3,4)=-4.19
     (dimer (0,1) intacto; bond (3,4) reforzado; dimers (2,3)/(4,5)
      'derretidos' en superposicion -- firma de estado tipo spin-liquid)


In [4]:
"""
reproduce_19site_table1.py
----------------------------
Reproduce el clúster de 19 sitios ("two unit-cell Kagome lattice") usado en
la fila (d) de la Tabla 1 del paper de Ahsan, y verifica su energía exacta.

La lista de bonds fue recuperada de:
  1) El notebook original del autor (celda comentada "Two Kagome Lattice Waala")
  2) La presentación previa del programa de créditos IBM del mismo autor
     (slides 5-12), que describe explícitamente esta misma red de 19 sitios /
     30 edges, el dimer covering, y el triángulo defecto (1,2,11).

Resultados esperados:
    Exacto (ED):                          -29.14   (obtenido aquí: -29.1462)
    VQE sin calibrar <phi|H|phi>:          -25.7 a -27.2   (error 7-12%)
    VQE calibrado (J' en triangulo defecto): mejora sustancial del error
        (paper: -29.10 +/- 0.05, error 0.14%;
         slide con peso 1.5 en los 3 bonds del triangulo defecto: -29.2, 0.9%)

NOTA: el mínimo global exacto del VQE calibrado es sensible a la
inicialización de parámetros (el paper parte de un estado dimerizado ya
preparado, no de reinicios aleatorios) y al número de iteraciones del
optimizador clásico. Aquí se usan reinicios aleatorios + COBYLA con un
presupuesto de iteraciones moderado; para acercarse más al óptimo del paper,
aumentar n_restarts / maxiter (a costa de tiempo de cómputo: cada evaluación
de energía sobre 19 qubits toma bastante más que en el caso de 6 sitios).
"""

import time
from kagome_common import build_hamiltonian, exact_gs_energy, vqe_energy

n = 19

# Lista completa de bonds (30 edges) del cluster de 19 sitios
meas_lst = [[0, 1], [1, 2], [2, 3], [2, 4], [3, 4], [4, 5], [4, 6], [5, 6],
            [6, 7], [6, 8], [7, 8], [8, 9], [8, 10], [9, 10], [10, 11],
            [10, 12], [11, 12], [12, 13], [12, 14], [13, 14], [14, 15],
            [14, 16], [15, 16], [16, 17], [17, 18], [16, 18], [1, 11],
            [2, 11], [0, 17], [1, 17]]

# --- 1) Hamiltoniano original, J=1 uniforme ---
bonds_uniform = [(i, j, 1) for (i, j) in meas_lst]
H = build_hamiltonian(n, bonds_uniform)

E_exact = exact_gs_energy(H)
print(f"Numero de bonds: {len(meas_lst)}   Numero de sitios: {n}")
print(f"Energia exacta del estado fundamental (ED, J=1 uniforme): {E_exact:.4f}")
print(f"  -> Paper / slide (Tabla 1, fila d): -29.14\n")

# --- 2) VQE sin calibrar ---
t0 = time.time()
val_u, theta_u, ansatz = vqe_energy(H, n, reps=1, n_restarts=2, seed=3,
                                     maxiter=400, method='COBYLA')
err_u = abs(val_u - E_exact) / abs(E_exact) * 100
print(f"VQE <phi|H|phi> (sin calibrar): {val_u:.4f}   error={err_u:.2f}%"
      f"   ({time.time()-t0:.0f}s)")
print("  -> Paper/slide: -25.7 a -27.2, error 7-12%\n")

# --- 3) Hamiltoniano calibrado ---
# Triangulo defecto identificado explicitamente en la presentacion del
# programa de creditos: S = {(1,2), (1,11), (2,11)}, con peso (1+wt), wt=0.5
S = [(1, 2), (1, 11), (2, 11)]
bonds_pert = [(i, j, 1.5 if (i, j) in S else 1.0) for (i, j) in meas_lst]
H_pert = build_hamiltonian(n, bonds_pert)

t0 = time.time()
val_p, theta_p, ansatz_p = vqe_energy(H_pert, n, reps=1, n_restarts=2, seed=21,
                                       maxiter=400, method='COBYLA')
err_p = abs(val_p - E_exact) / abs(E_exact) * 100
print(f"VQE <phi|H_pert|phi> (peso 1.5 en triangulo defecto S): "
      f"{val_p:.4f}   error={err_p:.2f}%   ({time.time()-t0:.0f}s)")
print("  -> Paper: -29.10, error 0.14%  /  slide: -29.2, error 0.9%")
print("\n(La calibracion reduce el error de forma sustancial en ambos casos;")
print(" el valor exacto del minimo depende de la inicializacion/presupuesto")
print(" de optimizacion -- ver nota al inicio del archivo.)")


Numero de bonds: 30   Numero de sitios: 19
Energia exacta del estado fundamental (ED, J=1 uniforme): -29.1462
  -> Paper / slide (Tabla 1, fila d): -29.14

VQE <phi|H|phi> (sin calibrar): -25.6653   error=11.94%   (306s)
  -> Paper/slide: -25.7 a -27.2, error 7-12%

VQE <phi|H_pert|phi> (peso 1.5 en triangulo defecto S): -26.6577   error=8.54%   (303s)
  -> Paper: -29.10, error 0.14%  /  slide: -29.2, error 0.9%

(La calibracion reduce el error de forma sustancial en ambos casos;
 el valor exacto del minimo depende de la inicializacion/presupuesto
 de optimizacion -- ver nota al inicio del archivo.)


# Heisenberg Gate

La compuerta de Heisenberg se define como

\begin{equation*}
G(\theta)=\exp\left[-i\theta\left(XX+YY+ZZ\right)\right].
\end{equation*}

La referencia exacta para esta operación corresponde a la exponencial de la matriz, la cual puede calcularse mediante la función \textit{scipy.linalg.expm}.

Debido a que los operadores \(XX\), \(YY\) y \(ZZ\) conmutan entre sí dos a dos en un sistema de dos qubits, la exponencial puede factorizarse exactamente como

\begin{equation*}
\exp\left[-i\theta\left(XX+YY+ZZ\right)\right]
=
\exp\left(-i\theta XX\right)
\exp\left(-i\theta YY\right)
\exp\left(-i\theta ZZ\right).
\end{equation*}

En consecuencia, la implementación mediante puertas nativas del circuito cuántico es

\begin{equation*}
G(\theta)
=
R_{XX}(2\theta)\,
R_{YY}(2\theta)\,
R_{ZZ}(2\theta).
\end{equation*}

Esta descomposición es \textbf{exacta} y no requiere una aproximación de Trotter, precisamente porque los operadores \(XX\), \(YY\) y \(ZZ\) conmutan entre sí en el espacio de Hilbert de dos qubits.

In [5]:
import numpy as np
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.circuit.library import RXXGate, RYYGate, RZZGate
from qiskit.quantum_info import Operator, Statevector

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
I2 = np.eye(2, dtype=complex)

def heisenberg_hamiltonian_matrix():
    """XX + YY + ZZ como matriz 4x4 (referencia)."""
    return np.kron(X, X) + np.kron(Y, Y) + np.kron(Z, Z)

def heisenberg_gate_matrix_exact(theta):
    """G(theta) via exponencial de matriz -- la 'verdad' contra la que validamos."""
    H = heisenberg_hamiltonian_matrix()
    return expm(-1j * theta * H)

def heisenberg_gate_circuit(theta, label=None):
    """G(theta) como circuito de 2 qubits en compuertas nativas de Qiskit."""
    qc = QuantumCircuit(2, name=label or f"G({theta:.3f})")
    qc.append(RXXGate(2 * theta), [0, 1])
    qc.append(RYYGate(2 * theta), [0, 1])
    qc.append(RZZGate(2 * theta), [0, 1])
    return qc

def heisenberg_gate(theta):
    """Devuelve la puerta como Gate reusable (para insertar en circuitos grandes)."""
    return heisenberg_gate_circuit(theta).to_gate(label=f"G({theta:.3f})")


# ---------------------------------------------------------------------
# Validacion
# ---------------------------------------------------------------------
def validate(theta, verbose=True):
    results = {}

    # 1) Coincidencia exacta de la matriz unitaria (circuito vs. exponencial)
    U_exact = heisenberg_gate_matrix_exact(theta)
    U_circ = Operator(heisenberg_gate_circuit(theta)).data
    # Comparar hasta fase global (por si el circuito difiere en fase, aunque
    # aqui no deberia, ya que RXX/RYY/RZZ estan definidas sin fase global extra)
    diff = np.max(np.abs(U_exact - U_circ))
    results['max_matrix_diff'] = diff
    results['unitary_match'] = diff < 1e-10

    # 2) Unitariedad del circuito
    unit_err = np.max(np.abs(U_circ.conj().T @ U_circ - np.eye(4)))
    results['unitarity_error'] = unit_err

    # 3) Autovalores fisicos: singlete vs triplete
    singlet = np.array([0, 1, -1, 0], dtype=complex) / np.sqrt(2)      # (|01>-|10>)/sqrt2
    triplet0 = np.array([1, 0, 0, 0], dtype=complex)                    # |00>
    triplet1 = np.array([0, 1, 1, 0], dtype=complex) / np.sqrt(2)      # (|01>+|10>)/sqrt2
    triplet2 = np.array([0, 0, 0, 1], dtype=complex)                    # |11>

    expected_singlet_phase = np.exp(3j * theta)     # eigenvalue de H es -3
    expected_triplet_phase = np.exp(-1j * theta)    # eigenvalue de H es +1

    out_s = U_circ @ singlet
    out_t0 = U_circ @ triplet0
    out_t1 = U_circ @ triplet1
    out_t2 = U_circ @ triplet2

    def phase_error(vec_out, vec_in, expected_phase):
        # vec_out deberia ser expected_phase * vec_in
        return np.max(np.abs(vec_out - expected_phase * vec_in))

    results['singlet_phase_error'] = phase_error(out_s, singlet, expected_singlet_phase)
    results['triplet0_phase_error'] = phase_error(out_t0, triplet0, expected_triplet_phase)
    results['triplet1_phase_error'] = phase_error(out_t1, triplet1, expected_triplet_phase)
    results['triplet2_phase_error'] = phase_error(out_t2, triplet2, expected_triplet_phase)

    if verbose:
        print(f"--- Validacion de G(theta={theta:.4f}) ---")
        print(f"  |U_exacta - U_circuito|_max        = {diff:.2e}   "
              f"{'OK' if results['unitary_match'] else 'FALLO'}")
        print(f"  Error de unitariedad (U^dag U - I)  = {unit_err:.2e}")
        print(f"  Fase en singlete  (esperado e^{{3i*theta}}) : error = "
              f"{results['singlet_phase_error']:.2e}")
        print(f"  Fase en triplete |00>  (e^{{-i*theta}})    : error = "
              f"{results['triplet0_phase_error']:.2e}")
        print(f"  Fase en triplete (|01>+|10>)/sqrt2         : error = "
              f"{results['triplet1_phase_error']:.2e}")
        print(f"  Fase en triplete |11>                      : error = "
              f"{results['triplet2_phase_error']:.2e}")
    return results


if __name__ == "__main__":
    for theta in [0.0, 0.3, 0.7853981633974483, 1.0, 2.5]:  # incluye pi/4
        validate(theta)
        print()

--- Validacion de G(theta=0.0000) ---
  |U_exacta - U_circuito|_max        = 0.00e+00   OK
  Error de unitariedad (U^dag U - I)  = 0.00e+00
  Fase en singlete  (esperado e^{3i*theta}) : error = 0.00e+00
  Fase en triplete |00>  (e^{-i*theta})    : error = 0.00e+00
  Fase en triplete (|01>+|10>)/sqrt2         : error = 0.00e+00
  Fase en triplete |11>                      : error = 0.00e+00

--- Validacion de G(theta=0.3000) ---
  |U_exacta - U_circuito|_max        = 2.48e-16   OK
  Error de unitariedad (U^dag U - I)  = 4.44e-16
  Fase en singlete  (esperado e^{3i*theta}) : error = 1.57e-16
  Fase en triplete |00>  (e^{-i*theta})    : error = 0.00e+00
  Fase en triplete (|01>+|10>)/sqrt2         : error = 1.24e-16
  Fase en triplete |11>                      : error = 0.00e+00

--- Validacion de G(theta=0.7854) ---
  |U_exacta - U_circuito|_max        = 1.70e-16   OK
  Error de unitariedad (U^dag U - I)  = 4.51e-33
  Fase en singlete  (esperado e^{3i*theta}) : error = 1.57e-16
  Fase en